# LAB: Workflows avec LangGraph
## Building Workflows with LangGraph Graphs
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre la création de workflows variés avec LangGraph.

## 📚 Qu'est-ce qu'un Workflow LangGraph?

Un workflow est un graphe dirigé où:
- 📦 **Nodes** = Fonctions qui traitent l'état
- 🔗 **Edges** = Connexions entre nodes
- 📊 **State** = Données partagées
- 🎯 **START/END** = Points clés

### Types de Workflows:
1. **Linéaire**: A → B → C → END
2. **Conditionnel**: A → (B ou C) → END
3. **Boucle**: A → ?(condition) → A ou END
4. **Parallèle**: A → (B + C) → D → END

## ⚙️ SETUP

In [ ]:
print("""
🚀 INSTALLATION LANGGRAPH
="*70

uv init
uv venv
.venv\Scripts\activate  # Windows
source .venv/bin/activate  # macOS/Linux

uv add langgraph langchain langchain-core

Vérifier:
  python --version
  uv --version
""")

## 1️⃣ PARTIE 1: Hello Graph (Simple)

In [ ]:
print("""
1️⃣ HELLO GRAPH - WORKFLOW LE PLUS SIMPLE
="*70

""")

print("""
# hello_graph.py
from langgraph.graph import StateGraph, MessagesState, START, END

def hello_node(state: MessagesState):
    """Un node qui ajoute un message."""
    return {"messages": [{"role": "ai", "content": "Hello World"}]}

# Construire le graphe
builder = StateGraph(MessagesState)
builder.add_node("hello", hello_node)
builder.add_edge(START, "hello")
builder.add_edge("hello", END)

# Compiler et exécuter
graph = builder.compile()
result = graph.invoke({"messages": [{"role": "user", "content": "Hi"}]})
print(result["messages"][-1].content)
# → "Hello World"
""")

print("\nFlux:")
print("  START → hello_node() → END")
print()
print("État (MessagesState):")
print("  • messages: List de messages")
print("  • Reducer: add (concaténation)")

## 2️⃣ PARTIE 2: Two-Step Workflow

In [ ]:
print("""
2️⃣ TWO-STEP WORKFLOW
="*70

# État personnalisé
from typing_extensions import TypedDict

class State(TypedDict):
    topic: str
    joke: str

# Nodes
def refine_topic(state: State):
    return {"topic": state["topic"] + " (and cats)"}

def write_joke(state: State):
    return {"joke": f"Here is a joke about {state['topic']}."}

# Graphe
builder = StateGraph(State)
builder.add_node("refine_topic", refine_topic)
builder.add_node("write_joke", write_joke)
builder.add_edge(START, "refine_topic")
builder.add_edge("refine_topic", "write_joke")
builder.add_edge("write_joke", END)

# Exécution
graph = builder.compile()
result = graph.invoke({"topic": "ice cream", "joke": ""})
print(result)
# → {"topic": "ice cream (and cats)", "joke": "..."}
""")

print("\nFlux:")
print("  START → refine_topic() → write_joke() → END")
print()
print("État partagé:")
print("  • topic: Modifié par refine_topic")
print("  • joke: Rempli par write_joke")

## 3️⃣ PARTIE 3: Reducers (Fusionner les données)

In [ ]:
print("""
3️⃣ REDUCERS - FUSIONNER PLUSIEURS VALEURS
="*70

Problème: Si plusieurs nodes modifient le même champ,
comment fusionner les valeurs?

Solution: Reducers
  • add: Concaténer les listes
  • Ou fonction personnalisée

# État avec reducer
from typing_extensions import Annotated
from operator import add

class State(TypedDict):
    topic: str
    log: Annotated[list[str], add]  # ← add est le reducer

# Nodes modifient log
def step_a(state: State):
    return {"log": [f"step_a saw {state['topic']}"]}

def step_b(state: State):
    return {"log": [f"step_b finishing {state['topic']}"]}

def step_c(state: State):
    return {"log": [f"step_c at last {state['topic']}"]}

# Graphe
builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)
builder.add_node("step_c", step_c)
builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", "step_c")
builder.add_edge("step_c", END)

# Exécution
graph = builder.compile()
result = graph.invoke({"topic": "langgraph", "log": []})
print(result['log'])
# → ['step_a saw langgraph', 'step_b finishing langgraph', ...]
""")

print("\nReducers communs:")
print("  • add: Liste concaténée")
print("  • lambda a,b: a+b: Fusion personnalisée")
print("  • max/min: Valeur extrême")
print("  • lambda a,b: b: Prendre la dernière")

## 4️⃣ PARTIE 4: État avec Messages

In [ ]:
print("""
4️⃣ ÉTAT AVEC MESSAGES
="*70

Idéal pour les workflows conversationnels

# État personnalisé avec messages
from typing_extensions import Annotated
from operator import add
from langchain_core.messages import BaseMessage

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    steps: int

# Nodes qui retournent des messages
def echo_node(state: ChatState):
    return {
        "messages": [{"role": "ai", "content": f"Step {state['steps']}: got message."}],
        "steps": state["steps"] + 1,
    }

def echo_node_1(state: ChatState):
    return {
        "messages": [{"role": "ai", "content": f"Step {state['steps']}: got message 1."}],
        "steps": state["steps"] + 1,
    }

# Graphe
builder = StateGraph(ChatState)
builder.add_node("echo", echo_node)
builder.add_node("echo_1", echo_node_1)
builder.add_edge(START, "echo")
builder.add_edge("echo", "echo_1")
builder.add_edge("echo_1", END)

# Exécution
graph = builder.compile()
result = graph.invoke(
    {"messages": [{"role": "user", "content": "hello"}], "steps": 1}
)
print(result["messages"])
# → [HumanMessage(content="hello"), 
#     AIMessage(content="Step 1: got message."),
#     AIMessage(content="Step 2: got message 1.")]
""")

print("\nAvantages:")
print("  ✓ Format standardisé")
print("  ✓ Compatible avec LLMs")
print("  ✓ Historique complet")
print("  ✓ Métadonnées préservées")

## 5️⃣ PARTIE 5: Workflow Conditionnel

In [ ]:
print("""
5️⃣ WORKFLOW CONDITIONNEL
="*70

Routing basé sur une condition du state

# État
from typing import Literal

class State(TypedDict):
    topic: str
    joke: str
    improved: str

# Nodes
def generate_joke(state: State):
    return {"joke": f"A joke about {state['topic']} maybe ? "}

def improve_joke(state: State):
    return {"improved": state['joke'] + " !!!"}

# Fonction de condition (routing)
def check_joke(state: State) -> Literal["improve", "end"]:
    if "!" in state["joke"]:
        return "end"  # Blague déjà bonne
    return "improve"  # Améliorer

# Graphe
builder = StateGraph(State)
builder.add_node("generate_joke", generate_joke)
builder.add_node("improve_joke", improve_joke)
builder.add_edge(START, "generate_joke")
builder.add_conditional_edges(
    "generate_joke",
    check_joke,
    {"improve": "improve_joke", "end": END}
)
builder.add_edge("improve_joke", END)

# Exécution
graph = builder.compile()
result = graph.invoke({"topic": "cats", "joke": "", "improved": ""})
print(result)
""")

print("\nFlux:")
print("  START")
print("    ↓")
print("  generate_joke()")
print("    ↓ (check_joke?)")
print("    ├─ "!" in joke → END")
print("    └─ "!" not in joke → improve_joke() → END")

## 6️⃣ PARTIE 6: Workflow en Boucle

In [ ]:
print("""
6️⃣ WORKFLOW EN BOUCLE
="*70

Répéter un node jusqu'à condition

# État
class State(TypedDict):
    n: int
    log: list[str]

# Node qui incrémente
def step(state: State):
    n = state["n"] + 1
    return {"n": n, "log": state["log"] + [f"n is now {n}"]}

# Condition d'arrêt
def should_continue(state: State) -> Literal["again", "stop"]:
    return "again" if state["n"] < 5 else "stop"

# Graphe avec boucle
builder = StateGraph(State)
builder.add_node("step", step)
builder.add_edge(START, "step")
builder.add_conditional_edges(
    "step",
    should_continue,
    {"again": "step", "stop": END}  # ← Boucle ici!
)

# Exécution
graph = builder.compile()
result = graph.invoke({"n": 0, "log": []})
print(result['log'])
# → ['n is now 1', 'n is now 2', ..., 'n is now 5']
print(f"Final n: {result['n']}")  # → 5
""")

print("\nFlux d'exécution:")
print("  START")
print("    ↓")
print("  step() n=1")
print("    ↓ should_continue?")
print("  n<5 → step() n=2")
print("    ↓ should_continue?")
print("  n<5 → step() n=3")
print("    ↓ should_continue?")
print("  n<5 → step() n=4")
print("    ↓ should_continue?")
print("  n<5 → step() n=5")
print("    ↓ should_continue?")
print("  n=5 → END")
print()
print("Cas d'usage:")
print("  • Itérations jusqu'à convergence")
print("  • Refiner query until acceptable")
print("  • Multi-round interactions")
print("  • Retry logic with state tracking")

## 📊 Résumé des 6 Patterns

In [ ]:
print("""
📊 LES 6 PATTERNS LANGGRAPH
="*70

1️⃣ HELLO GRAPH (Trivial)
   A → END
   Utilisation: Apprentissage, test

2️⃣ TWO-STEP (Linéaire)
   A → B → END
   Utilisation: Pipelines simples

3️⃣ REDUCERS (Fusion état)
   Annotated[list, add]
   Utilisation: Accumuler résultats

4️⃣ MESSAGES (État conversationnel)
   messages: Annotated[list[Message], add]
   Utilisation: Chatbots, assistants

5️⃣ CONDITIONNEL (Routing)
   A → (B ou C) → END
   Utilisation: Décisions, branches

6️⃣ BOUCLE (Itération)
   A → (A ou END)?
   Utilisation: Retry, refinement

🎯 CONSTRUCTION GÉNÉRALE:

1. Définir State (TypedDict + reducers)
2. Créer Nodes (fonctions)
3. Construire Builder (StateGraph)
4. Ajouter nodes (add_node)
5. Ajouter edges (add_edge)
6. Ajouter conditional edges si besoin
7. Compiler (compile())
8. Invoquer (invoke() ou stream())
""")